# 2D-to-3D Game Models — Hunyuan3D-2.1 on Colab T4

**Hunyuan3D-2.1** (Tencent, Apache 2.0) shape generation + our PBR texture pipeline.

**How to use:** Runtime → **T4 GPU** → **Run All** twice (1st installs + restarts, 2nd runs)

**Best input:** Single object, centered, 3/4 view, clean/white background, 512px+ PNG

In [ ]:
#@title 1. Install Hunyuan3D-2.1 + pipeline (~5 min first time)
import os, sys, subprocess

REPO_DIR = '/content/2d-to-3d-game-models'
HY3D_DIR = '/content/Hunyuan3D-2.1'
MARKER = '/content/.hy3d_installed_v3'

if not os.path.exists(MARKER):
    print('=== Installing Hunyuan3D-2.1 shape pipeline (~5 min) ===')
    os.chdir('/content')

    # Clone our pipeline repo
    subprocess.run(['rm', '-rf', REPO_DIR])
    subprocess.check_call(['git', 'clone', '-b',
        'claude/image-to-3d-pipeline-CnSII',
        'https://github.com/pmikola/2d-to-3d-game-models.git'])
    print('Pipeline repo cloned.')

    # Clone Hunyuan3D-2.1
    if not os.path.exists(HY3D_DIR):
        subprocess.check_call(['git', 'clone',
            'https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git',
            HY3D_DIR])
    print('Hunyuan3D-2.1 repo cloned.')

    # DO NOT install their full requirements.txt — it has bpy, deepspeed,
    # cupy, open3d, realesrgan etc. that crash on Colab.
    # Install ONLY what hy3dshape needs for shape generation:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'scipy', 'onnxruntime-gpu'])

    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'transformers', 'diffusers', 'accelerate', 'safetensors',
        'huggingface_hub', 'einops', 'omegaconf', 'pyyaml',
        'trimesh', 'pymeshlab', 'pygltflib', 'xatlas',
        'Pillow', 'opencv-python', 'imageio', 'scikit-image',
        'tqdm', 'ninja', 'pybind11', 'timm'])

    # rembg for background removal (pinned for Colab numpy compat)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        '--no-deps', 'rembg==2.0.57'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'pooch', 'pymatting', 'filetype', 'imagehash'])

    print('All deps installed.')

    open(MARKER, 'w').write('done')
    print('\n=== Done! Restarting kernel... ===')
    print('>>> Click Run All again after restart <<<')
    try:
        import IPython
        IPython.get_ipython().kernel.do_shutdown(True)
    except Exception:
        os._exit(0)

else:
    print('=== Already installed, skipping ===')
    os.chdir(REPO_DIR)
    import torch
    if torch.cuda.is_available():
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f'GPU: {torch.cuda.get_device_name(0)} ({vram:.1f} GB)')
    print('=== Ready! ===')

In [ ]:
#@title 2. Upload your image
from google.colab import files
from PIL import Image
from IPython.display import display

INPUT_PATH = '/content/test_input.png'

print('Upload PNG/JPG (single object, centered, clean background):')
uploaded = files.upload()

if uploaded:
    fname = list(uploaded.keys())[0]
    import shutil
    shutil.copy(fname, INPUT_PATH)
    img = Image.open(INPUT_PATH)
    print(f'Uploaded: {fname} ({img.size[0]}x{img.size[1]})')
    display(img.resize((300, 300)))
else:
    print('No upload. Using placeholder.')
    import numpy as np
    arr = np.full((512, 512, 3), 220, dtype=np.uint8)
    y, x = np.ogrid[-256:256, -256:256]
    arr[x**2 + y**2 < 150**2] = [180, 80, 40]
    Image.fromarray(arr).save(INPUT_PATH)

In [ ]:
#@title 3. Generate 3D: Hunyuan3D shape → Repair → UV → PBR → GLB
import os, sys, time, shutil, tempfile, base64
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from IPython.display import display, HTML

import logging
logging.basicConfig(level=logging.INFO,
    format='%(H:%M:%S)s [%(levelname)s] %(message)s')

HY3D_DIR = '/content/Hunyuan3D-2.1'
REPO_DIR = '/content/2d-to-3d-game-models'
INPUT_PATH = '/content/test_input.png'
OUTPUT_GLB = '/content/output/model.glb'
os.makedirs('/content/output', exist_ok=True)

# Add paths
sys.path.insert(0, HY3D_DIR)
sys.path.insert(0, os.path.join(HY3D_DIR, 'hy3dshape'))
os.chdir(HY3D_DIR)

start = time.time()

# === Stage 1: Load Hunyuan3D-2.1 shape pipeline ===
print('[1/6] Loading Hunyuan3D-2.1 shape model (downloads ~5GB first time)...')
from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline
from hy3dshape.rembg import BackgroundRemover

pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2.1',
    subfolder='hunyuan3d-dit-v2-1',
    use_safetensors=True,
)
print(f'  Hunyuan3D loaded on {pipeline.device}')

# === Stage 2: Preprocess (background removal) ===
print('[2/6] Removing background...')
image = Image.open(INPUT_PATH).convert('RGBA')
rembg = BackgroundRemover()
image = rembg(image)
print(f'  Preprocessed: {image.size}')
display(image.resize((200, 200)))

# === Stage 3: Generate 3D mesh ===
print('[3/6] Generating 3D geometry (~3-8 min on T4)...')
print('  This is REAL AI geometry — not a placeholder!')
mesh = pipeline(image=image, num_inference_steps=30)[0]

# Save raw mesh as intermediate
raw_glb = '/content/output/raw_shape.glb'
mesh.export(raw_glb)
print(f'  Generated: {len(mesh.vertices)} verts, {len(mesh.faces)} faces')

# Free GPU memory for texture stage
del pipeline, rembg
torch.cuda.empty_cache()
import gc; gc.collect()
print(f'  GPU memory freed.')

# Switch to our pipeline
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# === Stage 4: Mesh repair ===
print('[4/6] Repairing mesh (topology, smoothing, watertight)...')
from pipeline.mesh_repair import repair_and_prepare
from pipeline.geometry import normalize_mesh, unwrap_uvs, save_mesh_as_obj

import trimesh
mesh = trimesh.load(raw_glb, force='mesh')
mesh = repair_and_prepare(mesh, smooth_iterations=3)
normalize_mesh(mesh)
print(f'  Repaired: {len(mesh.vertices)} verts, {len(mesh.faces)} faces')

# === Stage 5: UV unwrap + PBR maps ===
print('[5/6] UV unwrapping + PBR maps...')
unwrap_uvs(mesh)

from pipeline.pbr_maps import generate_pbr_maps, save_pbr_maps
from pipeline.export import export_textured_dir_to_glb, validate_glb

texture_img = Image.open(INPUT_PATH).convert('RGB').resize((1024, 1024))

with tempfile.TemporaryDirectory() as tmp:
    obj_path = save_mesh_as_obj(mesh, tmp)
    textured = Path(tmp) / 'textured'
    textured.mkdir()

    texture_img.save(str(textured / 'texture_atlas.png'))
    shutil.copy(obj_path, str(textured / 'mesh_textured.obj'))

    pbr = generate_pbr_maps(texture_img, strength=1.5)
    save_pbr_maps(pbr, str(textured / 'pbr'))

    # Preview PBR
    row = Image.new('RGB', (256*4, 256))
    row.paste(texture_img.resize((256,256)), (0,0))
    row.paste(pbr['normal'].resize((256,256)), (256,0))
    row.paste(pbr['roughness'].convert('RGB').resize((256,256)), (512,0))
    row.paste(pbr['metallic'].convert('RGB').resize((256,256)), (768,0))
    print('  Albedo | Normal | Roughness | Metallic:')
    display(row)

    # === Stage 6: Export GLB ===
    print('[6/6] Exporting GLB with textures...')
    export_textured_dir_to_glb(str(textured), OUTPUT_GLB)

# Results
info = validate_glb(OUTPUT_GLB)
elapsed = time.time() - start
size_mb = os.path.getsize(OUTPUT_GLB) / (1024*1024)

print(f'\n{"="*60}')
print(f'DONE in {elapsed:.0f}s ({elapsed/60:.1f} min)')
print(f'  GLB: {OUTPUT_GLB} ({size_mb:.1f} MB)')
print(f'  Vertices: {info.get("total_vertices", "?")}')
print(f'  Faces: {info.get("total_faces", "?")}')
print(f'{"="*60}')

# Download button
print('\n')
with open(OUTPUT_GLB, 'rb') as f:
    b64 = base64.b64encode(f.read()).decode()
display(HTML(
    f'<h2><a href="data:model/gltf-binary;base64,{b64}" '
    f'download="model.glb" '
    f'style="background:#4CAF50;color:white;padding:15px 30px;'
    f'text-decoration:none;border-radius:8px;font-size:18px;">'
    f'📥 TAP TO DOWNLOAD model.glb ({size_mb:.1f} MB)</a></h2>'
))
# Also raw shape without textures
with open(raw_glb, 'rb') as f:
    b64_raw = base64.b64encode(f.read()).decode()
display(HTML(
    f'<a href="data:model/gltf-binary;base64,{b64_raw}" '
    f'download="raw_shape.glb" '
    f'style="color:#2196F3;font-size:14px;">'
    f'Also download raw shape (no texture)</a>'
))
print('\nView: https://gltf-viewer.donmccurdy.com/')